# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3896


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 133.62 GB
MemAvailable: 636.24 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face

## 2. SEML Pipeline

## 2.1 Response Generator

In [4]:
exp_id = "09-17-1-test"

## 2.2 Pipeline

### Set up the experiment

In [4]:
import logging
import os
from dotenv import load_dotenv
from huggingface_hub import login
import torch

from src.evaluations.evaluate_reliability import evaluate_reliability

# Set up logging
logger = logging.getLogger("quant_logger")
logger.setLevel(logging.INFO)

# Set up cache paths
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = os.path.join(CACHE_PATH, "hub")

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")

print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH
os.environ["TRANSFORMERS_CACHE"] = CACHE_PATH

torch.hub.set_dir(CACHE_PATH)

# Empty the cache
with torch.no_grad():
    torch.cuda.empty_cache()

# HuggingFace authentication
load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')
if huggingface_token is None:
    raise ValueError(
        f"Please set the HUGGINGFACE_TOKEN environment variable. "
        f"Looking in {os.path.join(os.getcwd(), '.env')}"
    )
else:
    print("Hugging Face token loaded successfully.")
login(token=huggingface_token, add_to_git_credential=True)

def run_evaluate(
    # Exp ID
    exp_id: str,
    save_excel: bool = True,
    num_excel_rows: int = 20,
    
    # Reliability dataset parameters
    seed=123,
    max_new_tokens=25,
    temperature=0.1,
    use_beam_search=False,
    strategy="Direct Completion",
    dataset_name="",
    taxonomy_type="0",
    typo_type="none",
    typo_intensity=0,
    n_repeats=10,
    n_beams=5,
    max_entries=None,
    
    # Model parameters
    seed_model=123,
    model_name="",
    model_path="",
    device="cuda",
    cache_path=CACHE_PATH
):
    ##################
    ## Print config ##
    ##################
    print("Received the following configuration:")
    print(f"  Seed: {seed}")
    print(f"  Max new tokens: {max_new_tokens}")
    print(f"  Temperature: {temperature}")
    print(f"  Use beam search: {use_beam_search}")
    print(f"  Strategy: {strategy}")
    print(f"  Dataset name: {dataset_name}")
    print(f"  Taxonomy type: {taxonomy_type}")
    print(f"  Number of repeats: {n_repeats}")
    print(f"  Number of beams: {n_beams}")
    print(f"  Max entries: {max_entries}")
    print(f"  Seed model: {seed_model}")
    print(f"  Model name: {model_name}")
    print(f"  Model path: {model_path}")
    print(f"  Device: {device}")

    results = evaluate_reliability(
        exp_id=exp_id,
        model_name=model_name,
        dataset_name=dataset_name,
        taxonomy_type=taxonomy_type,
        typo_type=typo_type,
        typo_intensity=typo_intensity,
        strategy=strategy,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        use_beam_search=use_beam_search,
        n_repeats=n_repeats,
        n_beams=n_beams,
        max_entries=max_entries,
        save_excel=save_excel,
        num_excel_rows=num_excel_rows,
        cache_dir=cache_path
    )

    return results

if __name__ == "__main__":
    # Example usage
    result = run_evaluate(
        exp_id="test-run",
        model_name="Llama-3-8B",
        dataset_name="P17",
        taxonomy_type="0",
        strategy="Direct Completion",
        max_entries=20,
    )
    print(result)

Setting cache path to /nfs/students/daro/.cache/huggingface
Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Received the following configuration:
  Seed: 123
  Max new tokens: 25
  Temperature: 0.1
  Use beam search: False
  Strategy: Direct Completion
  Dataset name: P17
  Taxonomy type: 0
  Number of repeats: 10
  Number of beams: 5
  Max entries: 20
  Seed model: 123
  Model name: Llama-3-8B
  Model path: 
  Device: cuda


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


  TOTAL: 1/200, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Completion, MAX_NEW_TOKENS: 25, RUN: 1/10
    IS_CORRECT: True, CLEANED: Answer: Eibenstock is located in the district of Zwickau in the Free State of Saxony in Germany, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 2/200, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Completion, MAX_NEW_TOKENS: 25, RUN: 2/10
    IS_CORRECT: True, CLEANED: Answer: Eibenstock is located in the district of Zwickau in the Free State of Saxony in Germany, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 3/200, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Completion, MAX_NEW_TOKENS: 25, RUN: 3/10
    IS_CORRECT: True, CLEANED: Answer: Eibenstock is located in the district of Zwickau in the Free State of Saxony in Germany, PROB: 1.00, ADJ_PROB: 1.00, ENTROPY: -0.00
  TOTAL: 4/200, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Direct Completion, MAX_NEW_TOKENS: 25, RUN: 4/10
    IS_CORRECT: True, CLEANED: Answer: Eibenstock is located in the dis

### Run configurations

In [ ]:
import itertools
from run_evaluate import run_evaluate  # Assuming the previous script is saved as run_evaluate.py

# Fixed parameters
fixed_params = {
    'exp_id': "taxonomy-test-09-27",
    'save_excel': True,
    'num_excel_rows': 200,
    'device': 'cuda',
    'seed': 42,
    'n_repeats': 5,
    'n_beams': 5,
    'max_entries': 5
}

# Grid parameters
grid_params = {
    'model_name': [
        'Llama-3-8B',
        # 'Bloomz',
        # 'GPT2-Large',
        # 'TinyLlama',
        # 'TinyLlama-Chat',
        # 'Llama-3-8B-AWQ-8bit',
        # 'Llama-3-8B-BNB-4bit',
        # 'Llama-3-8B-BNB-8bit',
        # 'Llama-3-8B-HQQ-8-uniform',
        # 'Llama-3-8B-HQQ-mixed',
        # 'Llama-3-8B-HQQ-LORA',
        # 'Llama-3-8B-QUANTO',
        # 'Llama-3-8B-QUANTO-CALIB',
        # 'Llama-3-8B-QUANTO-QAT',
        # 'Llama-3-8B-AQLM',
        # 'Llama-3-8B-AQLM-LORA'
    ],
    'max_new_tokens': [
        # 10,
        # 15,
        # 20,
        25,
        # 30,
        # 35,
        # 40,
        # 45,
        # 50
    ],
    'temperature': [0.1],
    'use_beam_search': [
        # True,
        False
    ],
    'strategy': [
        # "Original",
        # "Direct Answer",
        # "Factual Retrieval",
        # "Definitive Statement",
        # "Completion",
        # "Answer Completion",
        # "Fact Statement",
        # "Fill-in-the-Blank",
        # "Structured Answer Prompt",
        # "Direct Instruction",
        # "Contextual Prompts",
        # "Question-Answer Pairs",
        # "Q&A Format",
        # "Instructional",
        # "Summary",
        # "Echo",
        # "True Completion",
        "Direct Completion",
        # "Direct Query",
        # "First Thought",
        # "Deductive Reasoning",
        # "Expert Persona",
        # "Reflective Reasoning",
        # "Zero-Shot",
        # "One-Shot",
        # "Two-Shot",
        # "Three-Shot",
        # "Four-Shot",
        # "Five-Shot"
    ],
    'dataset_name': [
        # 'toy-qa-dataset',
        # 'P101',
        # 'P103',
        # 'P108',
        # 'P127',
        # 'P1376',
        # 'P1412',
        # 'P159',
        'P17',
        # 'P176',
        # 'P178',
        # 'P19',
        # 'P20',
        # 'P264',
        # 'P27',
        # 'P276',
        # 'P30',
        # 'P364',
        # 'P37',
        # 'P495',
        # 'P740'
    ],
    'taxonomy_type': [
        "0",
        "pos",
        "neg1",
        # "neg2",
        # "neg3",
        # "neg4",
        # "neg5"
    ],
    'typo_type': [
        "none",
        # "char_insertion",
        # "char_deletion",
        # "char_replacement",
        # "char_repetition",
        # "char_swapping",
        # "word_CMW",
        # "char_LCC",
        # "word_synonym",
        # "char_insert_noise",
        # "word_repeat",
        # "char_substitution",
        # "word_emoji",
        # "word_internet_slang",
        # "word_phrase_translation",
        # "word_context_aware_insertion",
        # "word_remove_punctuation",
        # "word_keyword_only",
        # "random"
    ],
    'typo_intensity': [
        0,
        # 1,
        # 2,
        # 3
    ]
}

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['model_name'],
    grid_params['max_new_tokens'],
    grid_params['temperature'],
    grid_params['use_beam_search'],
    grid_params['strategy'],
    grid_params['dataset_name'],
    grid_params['taxonomy_type'],
    grid_params['typo_type'],
    grid_params['typo_intensity']
))

# Run the evaluate function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    model_name, max_new_tokens, temperature, use_beam_search, strategy, dataset_name, taxonomy_type, typo_type, typo_intensity = combination

    # Print current combination details
    print(f"Running combination {i+1}/{len(grid_combinations)}")
    print(f"  Model Name: {model_name}")
    print(f"  Max New Tokens: {max_new_tokens}")
    print(f"  Temperature: {temperature}")
    print(f"  Use Beam Search: {use_beam_search}")
    print(f"  Strategy: {strategy}")
    print(f"  Dataset Name: {dataset_name}")
    print(f"  Taxonomy Type: {taxonomy_type}")
    print(f"  Typo Type: {typo_type}")
    print(f"  Typo Intensity: {typo_intensity}")

    result = run_evaluate(
        exp_id=fixed_params['exp_id'],
        save_excel=fixed_params['save_excel'],
        num_excel_rows=fixed_params['num_excel_rows'],
        seed=fixed_params['seed'],
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        use_beam_search=use_beam_search,
        strategy=strategy,
        dataset_name=dataset_name,
        taxonomy_type=taxonomy_type,
        typo_type=typo_type,
        typo_intensity=typo_intensity,
        n_repeats=fixed_params['n_repeats'],
        n_beams=fixed_params['n_beams'],
        max_entries=fixed_params['max_entries'],
        model_name=model_name,
        device=fixed_params['device']
    )

    # Append result with parameter details
    results.append({
        'result': result,
        'parameters': {
            'model_name': model_name,
            'max_new_tokens': max_new_tokens,
            'temperature': temperature,
            'use_beam_search': use_beam_search,
            'strategy': strategy,
            'dataset_name': dataset_name,
            'taxonomy_type': taxonomy_type,
            'typo_type': typo_type,
            'typo_intensity': typo_intensity
        }
    })

# Print or process the results as needed
for result in results:
    print(f"Parameters: {result['parameters']}")
    print(f"Results: {result['result']}")
    print("---")